In [2]:
# Clone repository
!git clone https://github.com/kienng221105/NLP_course
%cd NLP_course/lab03


fatal: destination path 'NLP_course' already exists and is not an empty directory.
/content/NLP_course/lab03


## 1. Imports và cấu hình

In [3]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 42.4 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import gzip
import json
import re
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab03':
    LAB_DIR = LAB_DIR / 'lab03'
PROJECT_DIR = LAB_DIR.parent
sys.path.insert(0, str(LAB_DIR))

import importlib
import cooccurrence as cooccurrence_module
cooccurrence_module = importlib.reload(cooccurrence_module)

build_vocabulary = cooccurrence_module.build_vocabulary
build_cooccurrence_matrix = cooccurrence_module.build_cooccurrence_matrix
cosine_similarity = cooccurrence_module.cosine_similarity
most_similar = cooccurrence_module.most_similar
build_cbow_examples = cooccurrence_module.build_cbow_examples
build_skipgram_pairs = cooccurrence_module.build_skipgram_pairs
context_evidence = cooccurrence_module.context_evidence

CORPUS_PATH = PROJECT_DIR / 'data' / 'c4-train.00000-of-01024-30K.json.gz'
COOCCURRENCE_MAX_VOCAB = None
MIN_COUNT = 2
SEED = 42
RESULT_RECORDS = []

def tokenize(text):
    return [
        match.group().lower().replace('’', "'")
        for match in re.finditer(r"[A-Za-z0-9]+(?:['’][A-Za-z0-9]+)?", text)
    ]

def tokenize_to_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|[\r\n]+", text)
    return [tokens for part in parts if (tokens := tokenize(part))]

def iter_documents():
    with gzip.open(CORPUS_PATH, 'rt', encoding='utf-8') as stream:
        for line in stream:
            record = json.loads(line)
            text = record.get('text', '')
            if isinstance(text, str):
                yield text

class C4SentenceCorpus:
    def __iter__(self):
        for text in iter_documents():
            yield from tokenize_to_sentences(text)

sentence_corpus = C4SentenceCorpus()


## 2. Đọc corpus
Notebook đọc toàn bộ file C4 trong data và dùng số document, câu, token thực tế được thống kê ở các cell sau.

In [5]:
document_count = sum(1 for _ in iter_documents())
sentence_count = 0
token_count = 0
for sentence in sentence_corpus:
    sentence_count += 1
    token_count += len(sentence)

corpus_vocabulary = build_vocabulary(sentence_corpus, min_count=1)
corpus_info = pd.DataFrame([{
    'T\u1ec7p corpus': CORPUS_PATH.name,
    'S\u1ed1 document': document_count,
    'S\u1ed1 c\u00e2u': sentence_count,
    'S\u1ed1 token': token_count,
    'Vocabulary th\u00f4': len(corpus_vocabulary),
}])
display(corpus_info)

for metric, value in {
    'documents': document_count,
    'sentences': sentence_count,
    'tokens': token_count,
    'vocabulary': len(corpus_vocabulary),
}.items():
    RESULT_RECORDS.append({
        'experiment': 'corpus_statistics',
        'corpus': CORPUS_PATH.name,
        'metric': metric,
        'score': value,
    })


,Tệp corpus,Số document,Số câu,Số token,Vocabulary thô
0,c4-train.00000-of-01024-30K.json.gz,30000,634941,10972699,196471


## 3. Tiền xử lý văn bản
Token hóa chữ thường, giữ từ chữ/số và apostrophe; tách câu theo dấu câu hoặc xuống dòng. Thay đổi cách tiền xử lý thì ghi lại và dùng nhất quán.

In [6]:
# Thống kê corpus được hiển thị ở cell trước.
display(corpus_info)


,Tệp corpus,Số document,Số câu,Số token,Vocabulary thô
0,c4-train.00000-of-01024-30K.json.gz,30000,634941,10972699,196471


## 4. Vocabulary cho co-occurrence
Dùng toàn corpus; ma trận CSR giúp lưu các giá trị khác 0 thay vì cấp phát ma trận đặc. min_count được ghi nhận trong bảng.

In [9]:
cooccurrence_corpus = sentence_corpus
cooccurrence_vocabulary = build_vocabulary(
    cooccurrence_corpus,
    min_count=MIN_COUNT,
    max_size=COOCCURRENCE_MAX_VOCAB,
)
RESULT_RECORDS.append({
    'experiment': 'cooccurrence',
    'corpus': CORPUS_PATH.name,
    'representation': 'word-context',
    'metric': 'vocabulary_size',
    'vocabulary_size': len(cooccurrence_vocabulary),
    'score': len(cooccurrence_vocabulary),
})
display(pd.DataFrame([{
    'Số câu dùng cho ma trận': sentence_count,
    'min_count': MIN_COUNT,
    'Giới hạn vocabulary': COOCCURRENCE_MAX_VOCAB,
    'Vocabulary thực tế': len(cooccurrence_vocabulary),
}]))

,Số câu dùng cho ma trận,min_count,Giới hạn vocabulary,Vocabulary thực tế
0,634941,2,None,101687


## 5. Experiment 1 - Co-occurrence, window 1
Ma trận CSR lưu các giá trị khác 0. Mỗi hàng là từ đích; mỗi cột là từ context.

In [10]:
cooccurrence_matrices = {}
cooccurrence_summary = []

window_1_matrix = build_cooccurrence_matrix(
    cooccurrence_corpus, cooccurrence_vocabulary, window_size=1
)
cooccurrence_matrices[1] = window_1_matrix
window_1_info = {
    'Window': 1,
    'Rows': window_1_matrix.shape[0],
    'Columns': window_1_matrix.shape[1],
    'Non-zero': window_1_matrix.nnz,
}
cooccurrence_summary.append(window_1_info)
RESULT_RECORDS.append({
    'experiment': 'cooccurrence',
    'corpus': CORPUS_PATH.name,
    'representation': 'word-context',
    'window': 1,
    'vocabulary_size': len(cooccurrence_vocabulary),
    'metric': 'nonzero_entries',
    'matrix_rows': window_1_matrix.shape[0],
    'matrix_columns': window_1_matrix.shape[1],
    'nonzero_entries': window_1_matrix.nnz,
    'score': window_1_matrix.nnz,
})
display(pd.DataFrame([window_1_info]))


,Window,Rows,Columns,Non-zero
0,1,101687,101687,4835875


### Similarity từ ma trận window 1

In [11]:
cooccurrence_pairs = [
    ('doctor', 'physician'),
    ('doctor', 'hospital'),
    ('cat', 'dog'),
]
window_1_similarity = []
for word_a, word_b in cooccurrence_pairs:
    if word_a in cooccurrence_vocabulary and word_b in cooccurrence_vocabulary:
        score = cosine_similarity(
            window_1_matrix.getrow(cooccurrence_vocabulary.index(word_a)),
            window_1_matrix.getrow(cooccurrence_vocabulary.index(word_b)),
        )
    else:
        score = np.nan
    window_1_similarity.append({
        'Cặp từ': f'{word_a} - {word_b}',
        'Cosine window 1': score,
    })
    RESULT_RECORDS.append({'experiment': 'cooccurrence_similarity',
        'corpus': CORPUS_PATH.name, 'window': 1,
        'metric': 'cosine_similarity',
        'item': f'{word_a} - {word_b}', 'score': score})
display(pd.DataFrame(window_1_similarity))


,Cặp từ,Cosine window 1
0,doctor - physician,0.913185
1,doctor - hospital,0.650607
2,cat - dog,0.868919


## 6. Experiment 1 - Co-occurrence, window 2

In [12]:
window_2_matrix = build_cooccurrence_matrix(
    cooccurrence_corpus, cooccurrence_vocabulary, window_size=2
)
cooccurrence_matrices[2] = window_2_matrix
window_2_info = {
    'Window': 2,
    'Rows': window_2_matrix.shape[0],
    'Columns': window_2_matrix.shape[1],
    'Non-zero': window_2_matrix.nnz,
}
cooccurrence_summary.append(window_2_info)
RESULT_RECORDS.append({
    'experiment': 'cooccurrence',
    'corpus': CORPUS_PATH.name,
    'representation': 'word-context',
    'window': 2,
    'vocabulary_size': len(cooccurrence_vocabulary),
    'metric': 'nonzero_entries',
    'matrix_rows': window_2_matrix.shape[0],
    'matrix_columns': window_2_matrix.shape[1],
    'nonzero_entries': window_2_matrix.nnz,
    'score': window_2_matrix.nnz,
})
display(pd.DataFrame([window_2_info]))


,Window,Rows,Columns,Non-zero
0,2,101687,101687,9305554


In [13]:
window_2_similarity = []
for word_a, word_b in cooccurrence_pairs:
    if word_a in cooccurrence_vocabulary and word_b in cooccurrence_vocabulary:
        score = cosine_similarity(
            window_2_matrix.getrow(cooccurrence_vocabulary.index(word_a)),
            window_2_matrix.getrow(cooccurrence_vocabulary.index(word_b)),
        )
    else:
        score = np.nan
    window_2_similarity.append({
        'Cặp từ': f'{word_a} - {word_b}',
        'Cosine window 2': score,
    })
    RESULT_RECORDS.append({'experiment': 'cooccurrence_similarity',
        'corpus': CORPUS_PATH.name, 'window': 2,
        'metric': 'cosine_similarity',
        'item': f'{word_a} - {word_b}', 'score': score})
display(pd.DataFrame(window_2_similarity))


,Cặp từ,Cosine window 2
0,doctor - physician,0.927676
1,doctor - hospital,0.752025
2,cat - dog,0.917972


## 7. Experiment 1 - Co-occurrence, window 5

In [14]:
window_5_matrix = build_cooccurrence_matrix(
    cooccurrence_corpus, cooccurrence_vocabulary, window_size=5
)
cooccurrence_matrices[5] = window_5_matrix
window_5_info = {
    'Window': 5,
    'Rows': window_5_matrix.shape[0],
    'Columns': window_5_matrix.shape[1],
    'Non-zero': window_5_matrix.nnz,
}
cooccurrence_summary.append(window_5_info)
RESULT_RECORDS.append({
    'experiment': 'cooccurrence',
    'corpus': CORPUS_PATH.name,
    'representation': 'word-context',
    'window': 5,
    'vocabulary_size': len(cooccurrence_vocabulary),
    'metric': 'nonzero_entries',
    'matrix_rows': window_5_matrix.shape[0],
    'matrix_columns': window_5_matrix.shape[1],
    'nonzero_entries': window_5_matrix.nnz,
    'score': window_5_matrix.nnz,
})
display(pd.DataFrame([window_5_info]))


,Window,Rows,Columns,Non-zero
0,5,101687,101687,19387434


In [15]:
window_5_similarity = []
for word_a, word_b in cooccurrence_pairs:
    if word_a in cooccurrence_vocabulary and word_b in cooccurrence_vocabulary:
        score = cosine_similarity(
            window_5_matrix.getrow(cooccurrence_vocabulary.index(word_a)),
            window_5_matrix.getrow(cooccurrence_vocabulary.index(word_b)),
        )
    else:
        score = np.nan
    window_5_similarity.append({
        'Cặp từ': f'{word_a} - {word_b}',
        'Cosine window 5': score,
    })
    RESULT_RECORDS.append({'experiment': 'cooccurrence_similarity',
        'corpus': CORPUS_PATH.name, 'window': 5,
        'metric': 'cosine_similarity',
        'item': f'{word_a} - {word_b}', 'score': score})
display(pd.DataFrame(window_5_similarity))
for (word_a, word_b), score in window_5_similarity and {
    (row['Cặp từ'].split(' - ')[0], row['Cặp từ'].split(' - ')[1]): row['Cosine window 5']
    for row in window_5_similarity
}.items():
    RESULT_RECORDS.append({
        'experiment': 'context_window_similarity', 'corpus': CORPUS_PATH.name,
        'window': 5, 'representation': 'cooccurrence',
        'metric': 'cosine_similarity', 'item': f'{word_a} - {word_b}', 'score': score,
    })


,Cặp từ,Cosine window 5
0,doctor - physician,0.947328
1,doctor - hospital,0.840242
2,cat - dog,0.954589


### So sánh co-occurrence giữa các window

In [16]:
cooccurrence_stats_table = pd.DataFrame(cooccurrence_summary)
cooccurrence_similarity_table = (
    pd.DataFrame(window_1_similarity)
    .merge(pd.DataFrame(window_2_similarity), on='Cặp từ')
    .merge(pd.DataFrame(window_5_similarity), on='Cặp từ')
)
display(cooccurrence_stats_table)
display(cooccurrence_similarity_table)


,Window,Rows,Columns,Non-zero
0,1,101687,101687,4835875
1,2,101687,101687,9305554
2,5,101687,101687,19387434


,Cặp từ,Cosine window 1,Cosine window 2,Cosine window 5
0,doctor - physician,0.913185,0.927676,0.947328
1,doctor - hospital,0.650607,0.752025,0.840242
2,cat - dog,0.868919,0.917972,0.954589


## 8. Core implementation - Top gần nhất

In [17]:
cooccurrence_neighbors = []
for query_word in ('doctor', 'cat'):
    if query_word not in cooccurrence_vocabulary:
        cooccurrence_neighbors.append({
            'Từ truy vấn': query_word,
            'Từ gần nhất': 'không có trong vocabulary',
            'Cosine': np.nan,
        })
        continue
    for neighbor, score in most_similar(
        query_word, window_1_matrix, cooccurrence_vocabulary, top_k=5
    ):
        cooccurrence_neighbors.append({
            'Từ truy vấn': query_word,
            'Từ gần nhất': neighbor,
            'Cosine': score,
        })
display(pd.DataFrame(cooccurrence_neighbors))


/content/NLP_course/lab03/cooccurrence.py:113: RuntimeWarning: invalid value encountered in sqrt
  row_norms = np.sqrt(np.asarray(matrix.multiply(matrix).sum(axis=1)).reshape(-1))


,Từ truy vấn,Từ gần nhất,Cosine
0,doctor,child,0.952152
1,doctor,dentist,0.924961
2,doctor,chinchilla,0.908002
3,doctor,dog,0.906647
4,doctor,vehicle,0.894524
5,cat,dog,0.868919
6,cat,vehicle,0.865378
7,cat,boat,0.864803
8,cat,message,0.864647
9,cat,mouse,0.856852


## 9. CBOW training pairs, window 1

In [18]:
toy_sentence = ['the', 'cat', 'eats', 'fish']
cbow_examples = build_cbow_examples(toy_sentence, window_size=1)
cbow_table = pd.DataFrame([
    {'Context': ', '.join(context), 'Target': target}
    for context, target in cbow_examples
])
display(cbow_table)


,Context,Target
0,cat,the
1,"the, eats",cat
2,"cat, fish",eats
3,eats,fish


## 10. Skip-gram training pairs, window 1

In [19]:
skipgram_pairs = build_skipgram_pairs(toy_sentence, window_size=1)
skipgram_table = pd.DataFrame([
    {'Target': target, 'Context': context}
    for target, context in skipgram_pairs
])
display(skipgram_table)


,Target,Context
0,the,cat
1,cat,the
2,cat,eats
3,eats,cat
4,eats,fish
5,fish,eats


## 11. Word2Vec dependency
Phần Word2Vec dùng Gensim. Nếu chưa có, cài gensim trong môi trường Python rồi khởi động lại kernel. CBOW dùng sg=0; Skip-gram dùng sg=1.

In [20]:
try:
    from gensim.models import Word2Vec
except ImportError:
    Word2Vec = None
    print('Cần cài gensim trước khi chạy các cell Word2Vec: pip install gensim')


### Hàm huấn luyện dùng chung
Dùng một worker và seed cố định để kết quả dễ tái lập hơn. Gensim dùng vector_size, window, min_count, epochs và sg để cấu hình Word2Vec.

In [21]:
def train_word2vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    epochs=10,
    objective='skipgram',
):
    if Word2Vec is None:
        raise ImportError('Cài gensim trước: pip install gensim')
    if objective not in ('cbow', 'skipgram'):
        raise ValueError("objective phải là 'cbow' hoặc 'skipgram'")
    started = time.perf_counter()
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=1,
        sg=1 if objective == 'skipgram' else 0,
        seed=SEED,
        epochs=epochs,
    )
    elapsed = time.perf_counter() - started
    parameter_bytes = model.wv.vectors.nbytes
    for attribute in ('syn1neg', 'syn1'):
        values = getattr(model, attribute, None)
        if isinstance(values, np.ndarray):
            parameter_bytes += values.nbytes
    size_mb = parameter_bytes / (1024 ** 2)
    return model, elapsed, size_mb


def mean_embedding(tokens, keyed_vectors):
    vectors = [keyed_vectors[word] for word in tokens if word in keyed_vectors]
    if not vectors:
        return None
    return np.mean(vectors, axis=0)


def semantic_search_top_score(model):
    query = tokenize('medical treatment')
    query_vector = mean_embedding(query, model.wv)
    if query_vector is None:
        return np.nan
    best_score = None
    for tokens in (tokenize(text) for text in iter_documents()):
        document_vector = mean_embedding(tokens, model.wv)
        if document_vector is None:
            continue
        score = cosine_similarity(query_vector, document_vector)
        best_score = score if best_score is None else max(best_score, score)
    return np.nan if best_score is None else float(best_score)


## 12. Word2Vec baseline - CBOW

In [ ]:
cbow_model, cbow_train_seconds, cbow_size_mb = train_word2vec(
    sentence_corpus,
    vector_size=100,
    window=5,
    min_count=MIN_COUNT,
    epochs=10,
    objective='cbow',
)
cbow_info = {
    'Objective': 'CBOW',
    'Vocabulary': len(cbow_model.wv),
    'Dimension': cbow_model.wv.vector_size,
    'Window': 5,
    'Min count': MIN_COUNT,
    'Epochs': 10,
    'Training time (s)': cbow_train_seconds,
    'Vector arrays (MB)': cbow_size_mb,
}
RESULT_RECORDS.append({
    'experiment': 'word2vec_objective',
    'corpus': CORPUS_PATH.name,
    'window': 5,
    'vector_size': 100,
    'min_count': MIN_COUNT,
    'epochs': 10,
    'objective': 'cbow',
    'metric': 'training_time_seconds',
    'training_time_seconds': cbow_train_seconds,
})
RESULT_RECORDS.append({
    'experiment': 'word2vec_objective', 'corpus': CORPUS_PATH.name,
    'vector_size': 100, 'window': 5, 'objective': 'cbow',
    'metric': 'model_size_mb', 'model_size_mb': cbow_size_mb,
})
display(pd.DataFrame([cbow_info]))


## 13. Word2Vec baseline - Skip-gram

In [ ]:
skipgram_model, skipgram_train_seconds, skipgram_size_mb = train_word2vec(
    sentence_corpus,
    vector_size=100,
    window=5,
    min_count=MIN_COUNT,
    epochs=10,
    objective='skipgram',
)
baseline_model = skipgram_model
skipgram_info = {
    'Objective': 'Skip-gram',
    'Vocabulary': len(skipgram_model.wv),
    'Dimension': skipgram_model.wv.vector_size,
    'Window': 5,
    'Min count': MIN_COUNT,
    'Epochs': 10,
    'Training time (s)': skipgram_train_seconds,
    'Vector arrays (MB)': skipgram_size_mb,
}
RESULT_RECORDS.append({
    'experiment': 'word2vec_objective',
    'corpus': CORPUS_PATH.name,
    'window': 5,
    'vector_size': 100,
    'min_count': MIN_COUNT,
    'epochs': 10,
    'objective': 'skipgram',
    'metric': 'training_time_seconds',
    'training_time_seconds': skipgram_train_seconds,
})
RESULT_RECORDS.append({
    'experiment': 'word2vec_objective', 'corpus': CORPUS_PATH.name,
    'vector_size': 100, 'window': 5, 'objective': 'skipgram',
    'metric': 'model_size_mb', 'model_size_mb': skipgram_size_mb,
})
display(pd.DataFrame([cbow_info, skipgram_info]))


## 14. Inspect embeddings - top 5
Các từ có min_count thấp hoặc không xuất hiện sẽ không có trong vocabulary; bảng ghi rõ thay vì dừng notebook.

In [ ]:
nearest_rows = []
for query in ('doctor', 'hospital', 'patient', 'disease',
              'computer', 'football', 'banana'):
    if query not in baseline_model.wv:
        nearest_rows.append({
            'Query': query,
            'Từ gần nhất': 'OOV / dưới min_count',
            'Similarity': np.nan,
        })
        continue
    for word, score in baseline_model.wv.most_similar(query, topn=5):
        nearest_rows.append({
            'Query': query,
            'Từ gần nhất': word,
            'Similarity': float(score),
        })
        RESULT_RECORDS.append({
            'experiment': 'nearest_words',
            'corpus': CORPUS_PATH.name,
            'objective': 'skipgram',
            'item': query,
            'related_word': word,
            'metric': 'cosine_similarity',
            'score': float(score),
            'rank': len([row for row in nearest_rows if row['Query'] == query]),
        })
display(pd.DataFrame(nearest_rows))


### Ngữ cảnh xuất hiện trong corpus
Dùng các từ context thường gặp làm bằng chứng để đọc kết quả nearest-neighbor.

In [ ]:
context_evidence_rows = []
for target in ('doctor', 'physician', 'hospital', 'patient', 'disease'):
    for context_word, count in context_evidence(
        sentence_corpus, target, window_size=2, top_k=8
    ):
        context_evidence_rows.append({
            'T\u1eeb \u0111\u00edch': target,
            'T\u1eeb context': context_word,
            'S\u1ed1 l\u1ea7n': count,
        })
display(pd.DataFrame(context_evidence_rows))
for row in context_evidence_rows:
    RESULT_RECORDS.append({
        'experiment': 'corpus_context_evidence',
        'corpus': CORPUS_PATH.name,
        'metric': 'context_count',
        'item': row['T\u1eeb \u0111\u00edch'],
        'related_word': row['T\u1eeb context'],
        'score': row['S\u1ed1 l\u1ea7n'],
    })


## 15. Experiment 3 - Context window 2

In [ ]:
window_2_model, window_2_train_seconds, window_2_size_mb = train_word2vec(
    sentence_corpus, vector_size=100, window=2,
    min_count=MIN_COUNT, epochs=10, objective='skipgram',
)
window_2_word_scores = {}
for word_a, word_b in cooccurrence_pairs:
    try:
        window_2_word_scores[(word_a, word_b)] = float(
            window_2_model.wv.similarity(word_a, word_b)
        )
    except KeyError:
        window_2_word_scores[(word_a, word_b)] = np.nan
RESULT_RECORDS.append({
    'experiment': 'context_window_model', 'corpus': CORPUS_PATH.name,
    'window': 2, 'vector_size': 100, 'min_count': MIN_COUNT,
    'epochs': 10, 'objective': 'skipgram',
    'metric': 'training_time_seconds', 'training_time_seconds': window_2_train_seconds,
})
for (word_a, word_b), score in window_2_word_scores.items():
    RESULT_RECORDS.append({
        'experiment': 'context_window_similarity', 'corpus': CORPUS_PATH.name,
        'window': 2, 'vector_size': 100, 'objective': 'skipgram',
        'metric': 'cosine_similarity', 'item': f'{word_a} - {word_b}', 'score': score,
    })
del window_2_model
display(pd.DataFrame([{
    'Window': 2,
    'Training time (s)': window_2_train_seconds,
    'Vector arrays (MB)': window_2_size_mb,
}]))


## 16. Experiment 3 - Context window 5

In [ ]:
window_5_word_scores = {}
for word_a, word_b in cooccurrence_pairs:
    try:
        window_5_word_scores[(word_a, word_b)] = float(
            baseline_model.wv.similarity(word_a, word_b)
        )
    except KeyError:
        window_5_word_scores[(word_a, word_b)] = np.nan
for (word_a, word_b), score in window_5_word_scores.items():
    RESULT_RECORDS.append({
        'experiment': 'context_window_similarity', 'corpus': CORPUS_PATH.name,
        'window': 5, 'vector_size': 100, 'objective': 'skipgram',
        'metric': 'cosine_similarity', 'item': f'{word_a} - {word_b}', 'score': score,
    })
display(pd.DataFrame([{
    'Window': 5,
    'Training time (s)': skipgram_train_seconds,
    'Vector arrays (MB)': skipgram_size_mb,
}]))


## 17. Experiment 3 - Context window 10

In [ ]:
window_10_model, window_10_train_seconds, window_10_size_mb = train_word2vec(
    sentence_corpus, vector_size=100, window=10,
    min_count=MIN_COUNT, epochs=10, objective='skipgram',
)
window_10_word_scores = {}
for word_a, word_b in cooccurrence_pairs:
    try:
        window_10_word_scores[(word_a, word_b)] = float(
            window_10_model.wv.similarity(word_a, word_b)
        )
    except KeyError:
        window_10_word_scores[(word_a, word_b)] = np.nan
RESULT_RECORDS.append({
    'experiment': 'context_window_model', 'corpus': CORPUS_PATH.name,
    'window': 10, 'vector_size': 100, 'min_count': MIN_COUNT,
    'epochs': 10, 'objective': 'skipgram',
    'metric': 'training_time_seconds', 'training_time_seconds': window_10_train_seconds,
})
for (word_a, word_b), score in window_10_word_scores.items():
    RESULT_RECORDS.append({
        'experiment': 'context_window_similarity', 'corpus': CORPUS_PATH.name,
        'window': 10, 'vector_size': 100, 'objective': 'skipgram',
        'metric': 'cosine_similarity', 'item': f'{word_a} - {word_b}', 'score': score,
    })
del window_10_model
window_rows = []
for pair in cooccurrence_pairs:
    window_rows.append({
        'Word pair': f'{pair[0]} - {pair[1]}',
        'Window 2': window_2_word_scores[pair],
        'Window 5': window_5_word_scores[pair],
        'Window 10': window_10_word_scores[pair],
    })
display(pd.DataFrame(window_rows))


## 18. Experiment 4 - Dimension 50

In [ ]:
dimension_50_model, dimension_50_seconds, dimension_50_size_mb = train_word2vec(
    sentence_corpus, vector_size=50, window=5,
    min_count=MIN_COUNT, epochs=10, objective='skipgram',
)
def evaluate_dimension_model(model):
    pair_score = (
        float(model.wv.similarity('doctor', 'physician'))
        if 'doctor' in model.wv and 'physician' in model.wv else np.nan
    )
    try:
        analogy = model.wv.most_similar(
            positive=['king', 'woman'], negative=['man'], topn=1
        )[0][0]
    except KeyError:
        analogy = 'OOV'
    retrieval_score = semantic_search_top_score(model)
    return pair_score, analogy, retrieval_score
dimension_50_similarity, dimension_50_analogy, dimension_50_retrieval = evaluate_dimension_model(dimension_50_model)
dimension_rows = [{
    'Dimension': 50,
    'Training time (s)': dimension_50_seconds,
    'Vector arrays (MB)': dimension_50_size_mb,
    'Cosine doctor-physician': dimension_50_similarity,
    'Analogy top-1': dimension_50_analogy,
    'Semantic search top cosine': dimension_50_retrieval,
}]
del dimension_50_model
display(pd.DataFrame(dimension_rows))


## 19. Experiment 4 - Dimension 100

In [ ]:
dimension_100_similarity, dimension_100_analogy, dimension_100_retrieval = evaluate_dimension_model(baseline_model)
dimension_rows.append({
    'Dimension': 100,
    'Training time (s)': skipgram_train_seconds,
    'Vector arrays (MB)': skipgram_size_mb,
    'Cosine doctor-physician': dimension_100_similarity,
    'Analogy top-1': dimension_100_analogy,
    'Semantic search top cosine': dimension_100_retrieval,
})
display(pd.DataFrame([dimension_rows[-1]]))


## 20. Experiment 4 - Dimension 300

In [ ]:
dimension_300_model, dimension_300_seconds, dimension_300_size_mb = train_word2vec(
    sentence_corpus, vector_size=300, window=5,
    min_count=MIN_COUNT, epochs=10, objective='skipgram',
)
dimension_300_similarity, dimension_300_analogy, dimension_300_retrieval = evaluate_dimension_model(dimension_300_model)
dimension_rows.append({
    'Dimension': 300,
    'Training time (s)': dimension_300_seconds,
    'Vector arrays (MB)': dimension_300_size_mb,
    'Cosine doctor-physician': dimension_300_similarity,
    'Analogy top-1': dimension_300_analogy,
    'Semantic search top cosine': dimension_300_retrieval,
})
del dimension_300_model
display(pd.DataFrame([dimension_rows[-1]]))


### So sánh các dimension

In [ ]:
display(pd.DataFrame(dimension_rows))
for row in dimension_rows:
    RESULT_RECORDS.append({
        'experiment': 'embedding_dimension',
        'corpus': CORPUS_PATH.name,
        'window': 5,
        'vector_size': row['Dimension'],
        'min_count': MIN_COUNT,
        'epochs': 10,
        'objective': 'skipgram',
        'metric': 'doctor_physician_cosine',
        'score': row['Cosine doctor-physician'],
        'training_time_seconds': row['Training time (s)'],
        'model_size_mb': row['Vector arrays (MB)'],
        'notes': f"analogy_top1={row['Analogy top-1']}",
    })
    RESULT_RECORDS.append({
        'experiment': 'embedding_dimension',
        'corpus': CORPUS_PATH.name,
        'window': 5,
        'vector_size': row['Dimension'],
        'min_count': MIN_COUNT,
        'epochs': 10,
        'objective': 'skipgram',
        'metric': 'semantic_search_top_cosine',
        'score': row['Semantic search top cosine'],
        'notes': 'Điểm cosine cao nhất trên truy vấn medical treatment',
    })


## 21. Evaluation - Word similarity

In [ ]:
word_pairs = [
    ('doctor', 'physician'),
    ('car', 'automobile'),
    ('king', 'queen'),
    ('cat', 'dog'),
    ('computer', 'banana'),
]
word_similarity_rows = []
for word_a, word_b in word_pairs:
    try:
        score = float(baseline_model.wv.similarity(word_a, word_b))
    except KeyError:
        score = np.nan
    word_similarity_rows.append({
        'Word pair': f'{word_a} - {word_b}',
        'Cosine similarity': score,
    })
word_similarity_table = pd.DataFrame(word_similarity_rows).sort_values(
    'Cosine similarity', ascending=False, na_position='last'
).reset_index(drop=True)
word_similarity_table['Rank'] = np.arange(1, len(word_similarity_table) + 1)
display(word_similarity_table)
for row in word_similarity_table.to_dict('records'):
    RESULT_RECORDS.append({
        'experiment': 'word_similarity',
        'corpus': CORPUS_PATH.name,
        'vector_size': 100,
        'window': 5,
        'objective': 'skipgram',
        'metric': 'cosine_similarity',
        'item': row['Word pair'],
        'score': row['Cosine similarity'],
        'rank': row['Rank'],
    })


## 22. Evaluation - Word analogy

In [ ]:
try:
    analogy_rows = baseline_model.wv.most_similar(
        positive=['king', 'woman'],
        negative=['man'],
        topn=5,
    )
    analogy_table = pd.DataFrame([
        {'Từ dự đoán': word, 'Similarity': float(score)}
        for word, score in analogy_rows
    ])
except KeyError as error:
    analogy_table = pd.DataFrame([{
        'Từ dự đoán': f'Từ thiếu trong vocabulary: {error}',
        'Similarity': np.nan,
    }])
display(analogy_table)
for rank, row in enumerate(analogy_table.to_dict('records'), start=1):
    RESULT_RECORDS.append({
        'experiment': 'word_analogy',
        'corpus': CORPUS_PATH.name,
        'vector_size': 100,
        'window': 5,
        'objective': 'skipgram',
        'metric': 'analogy_similarity',
        'related_word': row['Từ dự đoán'],
        'score': row['Similarity'],
        'rank': rank,
    })


## 23. Static Word2Vec và polysemy
Một model Word2Vec tĩnh tra cùng một vector cho từ bank bất kể câu nào. Cell này chỉ minh họa vector lookup; phần giải thích ý nghĩa ghi ở reflection.md.

In [ ]:
bank_contexts = [
    'I deposited money in the bank.',
    'We sat on the river bank.',
]
if 'bank' in baseline_model.wv:
    bank_vector_finance = baseline_model.wv['bank']
    bank_vector_river = baseline_model.wv['bank']
    bank_lookup_difference = float(np.linalg.norm(bank_vector_finance - bank_vector_river))
    bank_status = 'Cùng một lookup vector; không điều kiện theo câu'
else:
    bank_lookup_difference = np.nan
    bank_status = 'bank không có trong vocabulary'
display(pd.DataFrame([
    {'Câu': sentence, 'Vector của bank': bank_status, 'Lookup difference': bank_lookup_difference, 'Khoảng cách giữa hai lookup': bank_lookup_difference}
    for sentence in bank_contexts
]))


## 24. Application - Semantic search
Lấy trung bình vector của query words và document words đã có embedding; xếp hạng document bằng cosine. Đây là baseline đơn giản để quan sát semantic retrieval.

In [ ]:
semantic_query = tokenize('medical treatment')
query_vector = mean_embedding(semantic_query, baseline_model.wv)
semantic_results = []
if query_vector is not None:
    for document_index, document_text in enumerate(iter_documents()):
        document_vector = mean_embedding(
            tokenize(document_text), baseline_model.wv
        )
        if document_vector is None:
            continue
        score = cosine_similarity(query_vector, document_vector)
        semantic_results.append((document_index, score, document_text))
semantic_results.sort(key=lambda item: item[1], reverse=True)
top_documents = [
    {'Rank': rank, 'Document': text[:300], 'Cosine': score}
    for rank, (_, score, text) in enumerate(semantic_results[:5], start=1)
]
semantic_search_table = pd.DataFrame(top_documents)
display(semantic_search_table)
for row in semantic_search_table.to_dict('records'):
    RESULT_RECORDS.append({
        'experiment': 'semantic_search',
        'corpus': CORPUS_PATH.name,
        'vector_size': 100,
        'window': 5,
        'objective': 'skipgram',
        'metric': 'document_cosine',
        'item': 'medical treatment',
        'related_word': str(row['Rank']),
        'score': row['Cosine'],
        'rank': row['Rank'],
        'notes': row['Document'],
    })


## 25. Ghi lại kết quả
Cell cuối lưu các số đo đã tạo trong notebook vào results.csv. Tự hoàn thiện prediction.md, error_analysis.md và reflection.md sau khi xem kết quả; không điền nhận xét thay bạn.

In [ ]:
results_path = LAB_DIR / 'results.csv'
results_table = pd.DataFrame(RESULT_RECORDS)
results_table.to_csv(results_path, index=False)
display(pd.DataFrame([{
    'Tệp kết quả': results_path.name,
    'Số dòng': len(results_table),
}]))
